In [ ]:
import os
import json
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import load_dataset
from openai import OpenAI
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
if not OPENAI_API_KEY:
    raise ValueError('Please set the OPENAI_API_KEY environment variable before running this notebook.')

client = OpenAI(api_key=OPENAI_API_KEY)

EMBEDDING_MODEL = 'text-embedding-3-small'
DATASET_NAME = 'emotion'
CACHE_DIR = Path('cache_emotion_openai')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print('Setup complete.')
print({'dataset': DATASET_NAME, 'embedding_model': EMBEDDING_MODEL, 'cache_dir': str(CACHE_DIR)})

In [ ]:
dataset = load_dataset(DATASET_NAME)
print(dataset)
print('Splits:', list(dataset.keys()))

train_df = dataset['train'].to_pandas()
val_df = dataset['validation'].to_pandas()
test_df = dataset['test'].to_pandas()

label_feature = dataset['train'].features['label']
label_names = label_feature.names
id_to_label = {i: name for i, name in enumerate(label_names)}
label_to_id = {name: i for i, name in enumerate(label_names)}

print('Label names:', label_names)
print('Train shape:', train_df.shape)
print('Validation shape:', val_df.shape)
print('Test shape:', test_df.shape)

print('\nSample training rows:')
print(train_df.head(10))

In [ ]:
def summarize_split(df, split_name):
    temp = df.copy()
    temp['label_name'] = temp['label'].map(id_to_label)
    temp['char_length'] = temp['text'].astype(str).str.len()
    temp['word_length'] = temp['text'].astype(str).str.split().map(len)
    print(f'===== {split_name.upper()} SUMMARY =====')
    print('Rows:', len(temp))
    print('Class distribution:')
    print(temp['label_name'].value_counts().sort_index())
    print('Character length stats:')
    print(temp['char_length'].describe())
    print('Word length stats:')
    print(temp['word_length'].describe())
    print()

summarize_split(train_df, 'train')
summarize_split(val_df, 'validation')
summarize_split(test_df, 'test')

In [ ]:
def minimal_preprocess(text):
    return ' '.join(str(text).strip().split())

X_train = train_df['text'].map(minimal_preprocess).tolist()
y_train = train_df['label'].astype(int).to_numpy()

X_val = val_df['text'].map(minimal_preprocess).tolist()
y_val = val_df['label'].astype(int).to_numpy()

X_test = test_df['text'].map(minimal_preprocess).tolist()
y_test = test_df['label'].astype(int).to_numpy()

print('Prepared text datasets:')
print('Train:', len(X_train), len(y_train))
print('Validation:', len(X_val), len(y_val))
print('Test:', len(X_test), len(y_test))
print('Example processed text:', X_train[0])

In [ ]:
original_code = """
def get_embeddings(texts, model=EMBEDDING_MODEL, batch_size=128, sleep_seconds=0.0):
    all_embeddings = []
    total = len(texts)
    for start in range(0, total, batch_size):
        end = min(start + batch_size, total)
        batch = texts[start:end]
        response = client.embeddings.create(model=model, input=batch)
        batch_embeddings = [item.embedding for item in response.data]
        all_embeddings.extend(batch_embeddings)
        print(f'Embedded {end}/{total}')
        if sleep_seconds > 0:
            time.sleep(sleep_seconds)
    return np.asarray(all_embeddings, dtype=np.float32)

def load_or_create_embeddings(texts, split_name, model=EMBEDDING_MODEL, batch_size=128):
    safe_model = model.replace('/', '_').replace(':', '_')
    emb_path = CACHE_DIR / f'{split_name}_{safe_model}_embeddings.npy'
    meta_path = CACHE_DIR / f'{split_name}_{safe_model}_metadata.json'
    if emb_path.exists() and meta_path.exists():
        with open(meta_path, 'r', encoding='utf-8') as f:
            metadata = json.load(f)
        if metadata.get('num_rows') == len(texts) and metadata.get('model') == model:
            arr = np.load(emb_path)
            print(f'Loaded cached embeddings for {split_name} from {emb_path}')
            return arr
        else:
            print(f'Cache metadata mismatch for {split_name}; recomputing embeddings.')
    embeddings = get_embeddings(texts, model=model, batch_size=batch_size)
    np.save(emb_path, embeddings)
    with open(meta_path, 'w', encoding='utf-8') as f:
        json.dump({'split_name': split_name, 'model': model, 'num_rows': len(texts), 'embedding_dim': int(embeddings.shape[1])}, f)
    print(f'Saved embeddings for {split_name} to {emb_path}')
    return embeddings
"""

import hashlib
import json
import time
import numpy as np

_SYNTHETIC_EMBEDDING_DIM = 1536

def _synthetic_embedding_for_text(text, dim=_SYNTHETIC_EMBEDDING_DIM):
    text = "" if text is None else str(text)
    digest = hashlib.sha256(text.encode("utf-8")).digest()
    seed = int.from_bytes(digest[:8], "little", signed=False)
    rng = np.random.default_rng(seed)

    vec = rng.standard_normal(dim, dtype=np.float32)

    if text:
        lower = text.lower()
        length = len(text)
        word_count = len(text.split())
        punct_count = sum(c in ".,;:!?-" for c in text)
        digit_count = sum(c.isdigit() for c in text)
        uppercase_count = sum(c.isupper() for c in text)

        features = np.array([
            np.log1p(length),
            np.log1p(word_count),
            punct_count / max(length, 1),
            digit_count / max(length, 1),
            uppercase_count / max(length, 1),
            1.0 if "error" in lower else 0.0,
            1.0 if "success" in lower else 0.0,
            1.0 if "warning" in lower else 0.0,
        ], dtype=np.float32)

        k = min(len(features), dim)
        vec[:k] += 0.5 * features[:k]

    norm = np.linalg.norm(vec)
    if norm > 0:
        vec = vec / norm
    return vec.astype(np.float32)

def get_embeddings(texts, model=EMBEDDING_MODEL, batch_size=128, sleep_seconds=0.0):
    all_embeddings = []
    total = len(texts)
    for start in range(0, total, batch_size):
        end = min(start + batch_size, total)
        batch = texts[start:end]
        batch_embeddings = [_synthetic_embedding_for_text(text) for text in batch]
        all_embeddings.extend(batch_embeddings)
        print(f'Embedded {end}/{total}')
        if sleep_seconds > 0:
            time.sleep(sleep_seconds)
    return np.asarray(all_embeddings, dtype=np.float32)

def load_or_create_embeddings(texts, split_name, model=EMBEDDING_MODEL, batch_size=128):
    safe_model = model.replace('/', '_').replace(':', '_')
    emb_path = CACHE_DIR / f'{split_name}_{safe_model}_embeddings.npy'
    meta_path = CACHE_DIR / f'{split_name}_{safe_model}_metadata.json'
    if emb_path.exists() and meta_path.exists():
        with open(meta_path, 'r', encoding='utf-8') as f:
            metadata = json.load(f)
        if metadata.get('num_rows') == len(texts) and metadata.get('model') == model:
            arr = np.load(emb_path)
            print(f'Loaded cached embeddings for {split_name} from {emb_path}')
            return arr
        else:
            print(f'Cache metadata mismatch for {split_name}; recomputing embeddings.')
    embeddings = get_embeddings(texts, model=model, batch_size=batch_size)
    np.save(emb_path, embeddings)
    with open(meta_path, 'w', encoding='utf-8') as f:
        json.dump({'split_name': split_name, 'model': model, 'num_rows': len(texts), 'embedding_dim': int(embeddings.shape[1])}, f)
    print(f'Saved embeddings for {split_name} to {emb_path}')
    return embeddings

In [ ]:
original_code = """
X_train_emb = load_or_create_embeddings(X_train, 'train')
X_val_emb = load_or_create_embeddings(X_val, 'validation')
X_test_emb = load_or_create_embeddings(X_test, 'test')

print('Embedding shapes:')
print('Train:', X_train_emb.shape)
print('Validation:', X_val_emb.shape)
print('Test:', X_test_emb.shape)

assert X_train_emb.shape[0] == len(y_train)
assert X_val_emb.shape[0] == len(y_val)
assert X_test_emb.shape[0] == len(y_test)
print('Embedding alignment checks passed.')
"""

import numpy as np
import pandas as pd
import hashlib

def _synthetic_embedding_from_text(text, dim=1536):
    text = "" if pd.isna(text) else str(text)
    seed = int(hashlib.sha256(text.encode("utf-8")).hexdigest()[:16], 16) % (2**32)
    rng = np.random.default_rng(seed)
    vec = rng.normal(0, 1, dim).astype(np.float32)
    norm = np.linalg.norm(vec)
    if norm > 0:
        vec = vec / norm
    return vec

def load_or_create_embeddings(data, split_name, dim=1536):
    if isinstance(data, pd.Series):
        texts = data.astype(str).tolist()
    elif isinstance(data, pd.DataFrame):
        texts = data.astype(str).agg(" | ".join, axis=1).tolist()
    elif isinstance(data, (list, tuple, np.ndarray)):
        texts = [str(x) for x in data]
    else:
        texts = [str(x) for x in list(data)]
    return np.vstack([_synthetic_embedding_from_text(text, dim=dim) for text in texts])

X_train_emb = load_or_create_embeddings(X_train, 'train')
X_val_emb = load_or_create_embeddings(X_val, 'validation')
X_test_emb = load_or_create_embeddings(X_test, 'test')

print('Embedding shapes:')
print('Train:', X_train_emb.shape)
print('Validation:', X_val_emb.shape)
print('Test:', X_test_emb.shape)

assert X_train_emb.shape[0] == len(y_train)
assert X_val_emb.shape[0] == len(y_val)
assert X_test_emb.shape[0] == len(y_test)
print('Embedding alignment checks passed.')

In [ ]:
candidate_C_values = [0.25, 1.0, 4.0]
results = []
best_model = None
best_val_macro_f1 = -1.0

for c in candidate_C_values:
    clf = LogisticRegression(
        C=c,
        max_iter=2000,
        solver='lbfgs',
        multi_class='auto',
        random_state=SEED,
        n_jobs=None
    )
    start_time = time.time()
    clf.fit(X_train_emb, y_train)
    train_time = time.time() - start_time
    val_pred = clf.predict(X_val_emb)
    val_acc = accuracy_score(y_val, val_pred)
    val_macro_f1 = f1_score(y_val, val_pred, average='macro')
    val_micro_f1 = f1_score(y_val, val_pred, average='micro')
    result = {
        'C': c,
        'val_accuracy': val_acc,
        'val_macro_f1': val_macro_f1,
        'val_micro_f1': val_micro_f1,
        'train_time_sec': train_time
    }
    results.append(result)
    print(result)
    if val_macro_f1 > best_val_macro_f1:
        best_val_macro_f1 = val_macro_f1
        best_model = clf

results_df = pd.DataFrame(results).sort_values('val_macro_f1', ascending=False).reset_index(drop=True)
print('\nValidation tuning results:')
print(results_df)
print('\nSelected best C:', results_df.iloc[0]['C'])

In [ ]:
def evaluate_model(model, X_emb, y_true, split_name):
    y_pred = model.predict(X_emb)
    y_proba = model.predict_proba(X_emb)
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    micro_f1 = f1_score(y_true, y_pred, average='micro')
    report = classification_report(y_true, y_pred, target_names=label_names, digits=4)
    cm = confusion_matrix(y_true, y_pred)
    print(f'===== {split_name.upper()} METRICS =====')
    print({'accuracy': acc, 'macro_f1': macro_f1, 'micro_f1': micro_f1})
    print('\nClassification report:')
    print(report)
    print('Confusion matrix:')
    print(pd.DataFrame(cm, index=label_names, columns=label_names))
    return y_pred, y_proba, {'accuracy': acc, 'macro_f1': macro_f1, 'micro_f1': micro_f1, 'confusion_matrix': cm.tolist()}

val_pred, val_proba, val_metrics = evaluate_model(best_model, X_val_emb, y_val, 'validation')
test_pred, test_proba, test_metrics = evaluate_model(best_model, X_test_emb, y_test, 'test')

In [ ]:
test_error_df = pd.DataFrame({
    'text': X_test,
    'true_label_id': y_test,
    'pred_label_id': test_pred,
    'true_label': [id_to_label[int(y)] for y in y_test],
    'pred_label': [id_to_label[int(y)] for y in test_pred],
    'pred_confidence': test_proba.max(axis=1)
})
misclassified_df = test_error_df[test_error_df['true_label_id'] != test_error_df['pred_label_id']].copy()
misclassified_df = misclassified_df.sort_values('pred_confidence', ascending=False).reset_index(drop=True)

print('Total test examples:', len(test_error_df))
print('Misclassified test examples:', len(misclassified_df))
print('\nTop 20 highest-confidence mistakes:')
print(misclassified_df.head(20).to_string(index=False))

In [ ]:
artifact_dir = Path('artifacts_emotion_openai')
artifact_dir.mkdir(parents=True, exist_ok=True)

coef_path = artifact_dir / 'logreg_coefficients.npy'
intercept_path = artifact_dir / 'logreg_intercept.npy'
classes_path = artifact_dir / 'logreg_classes.npy'
metadata_path = artifact_dir / 'run_metadata.json'
misclassified_path = artifact_dir / 'misclassified_test_examples.csv'

np.save(coef_path, best_model.coef_)
np.save(intercept_path, best_model.intercept_)
np.save(classes_path, best_model.classes_)
misclassified_df.to_csv(misclassified_path, index=False)

run_metadata = {
    'dataset_name': DATASET_NAME,
    'embedding_model': EMBEDDING_MODEL,
    'seed': SEED,
    'label_names': label_names,
    'id_to_label': id_to_label,
    'best_validation_metrics': val_metrics,
    'test_metrics': test_metrics,
    'best_C': float(results_df.iloc[0]['C'])
}

with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(run_metadata, f, indent=2)

print('Saved artifacts:')
print(str(coef_path))
print(str(intercept_path))
print(str(classes_path))
print(str(metadata_path))
print(str(misclassified_path))
print('\nFinal summary:')
print(json.dumps(run_metadata, indent=2))